# EdgeGuard-Road Cityscapes Fine train preparation
Thin execution wrapper for EG-DATA-002. It verifies the approved archives, prepares train-only data in `/content`, promotes validated outputs to private Drive, and produces split candidates pending human approval. It does not train a model or access Cityscapes test labels, Fishyscapes, or SMIYC.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
RUN_ENV = {**os.environ, "PYTHONDONTWRITEBYTECODE": "1"}
REPOSITORY_URL = "https://github.com/emrealmaoglu/edgeguard-road.git"
BRANCH = "feat/first-vertical-slice"
EXPECTED_COMMIT = input("Reviewed EG-DATA-002 commit SHA: ").strip()
REPO_ROOT = Path("/content/edgeguard-road")
if len(EXPECTED_COMMIT) != 40 or any(c not in "0123456789abcdef" for c in EXPECTED_COMMIT):
    raise RuntimeError("Enter the reviewed lowercase 40-character commit SHA")
if REPO_ROOT.exists():
    raise RuntimeError("Start from a fresh Colab runtime")
subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY_URL, str(REPO_ROOT)],
    check=True,
    env=RUN_ENV,
)
actual_commit = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
    env=RUN_ENV,
).stdout.strip()
if actual_commit != EXPECTED_COMMIT:
    raise RuntimeError(f"Commit mismatch: expected {EXPECTED_COMMIT}, got {actual_commit}")
os.chdir(REPO_ROOT)

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[dev]"],
    cwd=REPO_ROOT,
    check=True,
    env=RUN_ENV,
)
subprocess.run(
    [sys.executable, "-m", "edgeguard", "doctor", "--json"],
    cwd=REPO_ROOT,
    check=True,
    env=RUN_ENV,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/unit/test_cityscapes_train.py",
        "tests/unit/test_prepare_cityscapes.py",
    ],
    cwd=REPO_ROOT,
    check=True,
    env=RUN_ENV,
)
status = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "status", "--porcelain=v1"],
    check=True,
    capture_output=True,
    text=True,
    env=RUN_ENV,
).stdout.strip()
if status:
    raise RuntimeError(f"Repository became dirty: {status}")

## Human-controlled Drive inputs
The two immutable source archives stay under `private_inputs/`; this notebook neither moves nor duplicates them inside Drive. Only the approved Cityscapes Fine train dataset and manifest destinations are created after validation.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
EXTERNAL_ROOT = Path("/content/drive/MyDrive/EdgeGuard")
os.environ["EDGEGUARD_EXTERNAL_ROOT"] = str(EXTERNAL_ROOT)
RUN_ENV["EDGEGUARD_EXTERNAL_ROOT"] = str(EXTERNAL_ROOT)
LEFT_ARCHIVE = EXTERNAL_ROOT / "private_inputs/leftImg8bit_trainvaltest.zip"
LABEL_ARCHIVE = EXTERNAL_ROOT / "private_inputs/gtFine_trainvaltest.zip"
DATASET_DESTINATION = EXTERNAL_ROOT / "datasets/cityscapes/fine/v1"
MANIFESTS_DESTINATION = EXTERNAL_ROOT / "manifests/cityscapes/fine/v1"
WORK_DIRECTORY = Path("/content/edgeguard-work/cityscapes-fine-v1")
VERIFY_ONLY = False
for archive in (LEFT_ARCHIVE, LABEL_ARCHIVE):
    if not archive.is_file():
        raise RuntimeError(f"Approved archive is missing: {archive.name}")
if not VERIFY_ONLY and (DATASET_DESTINATION.exists() or MANIFESTS_DESTINATION.exists()):
    raise RuntimeError("Destination exists; set VERIFY_ONLY=True only for exact validation")

In [ ]:
command = [
    sys.executable,
    "scripts/prepare_cityscapes.py",
    "--split",
    "train",
    "--left-images-archive",
    str(LEFT_ARCHIVE),
    "--labels-archive",
    str(LABEL_ARCHIVE),
    "--destination",
    str(DATASET_DESTINATION),
    "--manifests-destination",
    str(MANIFESTS_DESTINATION),
    "--work-directory",
    str(WORK_DIRECTORY),
    "--preparation-git-commit",
    EXPECTED_COMMIT,
    "--ontology-config",
    "configs/dataset/ontology_v1.yaml",
]
if VERIFY_ONLY:
    command.append("--verify-only")
completed = subprocess.run(
    command, cwd=REPO_ROOT, check=True, capture_output=True, text=True, env=RUN_ENV
)
completion = json.loads(completed.stdout)
if completion.get("selection_status") != "recommended_pending_human_approval":
    raise RuntimeError("Preparation did not preserve the human split-selection gate")
completion

In [ ]:
from google.colab import files

comparison = json.loads(
    (MANIFESTS_DESTINATION / "split_candidate_comparison.json").read_text(encoding="utf-8")
)
print(json.dumps(comparison, indent=2, sort_keys=True))
print("STOP: inspect candidates and obtain explicit human split approval before training.")
evidence_package = MANIFESTS_DESTINATION / completion["evidence_package_filename"]
files.download(str(evidence_package))